# Project Lab — Pretrain a Tiny GPT

This is the bridge from toy mechanics to a real pretraining campaign.

**Goal:** train a small decoder-only Transformer end-to-end on a public text corpus, with train/validation loss, checkpointing, generation, and resource measurements.

**Free-first:** designed for CPU sanity checks and free Colab GPU runs. Start with the default small configuration; increase it only after the pipeline works.

Learning loop: **predict → run → profile → break → interpret → scale**.


## 1. What we are actually building

```
raw text
  ↓
character tokenizer
  ↓
fixed-length token blocks
  ↓
decoder-only Transformer
  ↓
next-token logits
  ↓
cross-entropy
  ↓
AdamW
  ↓
checkpoint + validation
  ↓
generation
```

This is a real language-model training loop, only much smaller than a frontier model.


In [ ]:
import math, time, urllib.request
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## 2. Download the corpus

We use the public Tiny Shakespeare corpus as a compact teaching dataset. Record the URL and corpus hash in a research experiment if you use this result outside the course.


In [ ]:
url="https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
path=Path("tinyshakespeare.txt")
if not path.exists():
    urllib.request.urlretrieve(url, path)
text=path.read_text(encoding="utf-8")
print("characters:",len(text))
print(text[:300])


In [ ]:
chars=sorted(set(text))
vocab_size=len(chars)
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for ch,i in stoi.items()}
encode=lambda s:[stoi[c] for c in s]
decode=lambda ids:"".join(itos[i] for i in ids)
data=torch.tensor(encode(text),dtype=torch.long)
n=int(0.9*len(data))
train_data=data[:n]
val_data=data[n:]
print("vocab:",vocab_size,"train tokens:",len(train_data),"val tokens:",len(val_data))


## 3. Hyperparameters

Keep these small at first. The important skill is not memorizing a “correct” config; it is understanding how each setting affects memory, compute, optimization, and quality.


In [ ]:
batch_size=32
block_size=128
n_embd=128
n_head=4
n_layer=4
dropout=0.0
learning_rate=3e-4
max_steps=1000
eval_interval=100
eval_iters=20

def get_batch(split):
    source=train_data if split=="train" else val_data
    ix=torch.randint(len(source)-block_size,(batch_size,))
    x=torch.stack([source[i:i+block_size] for i in ix])
    y=torch.stack([source[i+1:i+block_size+1] for i in ix])
    return x.to(device),y.to(device)

xb,yb=get_batch("train")
print(xb.shape,yb.shape)


## 4. Causal self-attention

The key invariant is:

> Position t may use positions ≤ t, never future positions.

The implementation below uses standard scaled dot-product attention and an explicit causal mask.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self,n_embd,n_head,block_size):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head=n_head
        self.head_dim=n_embd//n_head
        self.qkv=nn.Linear(n_embd,3*n_embd)
        self.proj=nn.Linear(n_embd,n_embd)
        self.register_buffer("mask",torch.tril(torch.ones(block_size,block_size)).bool())

    def forward(self,x):
        B,T,C=x.shape
        q,k,v=self.qkv(x).chunk(3,dim=-1)
        q=q.view(B,T,self.n_head,self.head_dim).transpose(1,2)
        k=k.view(B,T,self.n_head,self.head_dim).transpose(1,2)
        v=v.view(B,T,self.n_head,self.head_dim).transpose(1,2)
        w=q@k.transpose(-2,-1)/math.sqrt(self.head_dim)
        w=w.masked_fill(~self.mask[:T,:T],float("-inf"))
        w=F.softmax(w,dim=-1)
        y=w@v
        y=y.transpose(1,2).contiguous().view(B,T,C)
        return self.proj(y)


In [ ]:
class MLP(nn.Module):
    def __init__(self,n_embd):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(n_embd,4*n_embd),
            nn.GELU(),
            nn.Linear(4*n_embd,n_embd),
        )
    def forward(self,x): return self.net(x)

class Block(nn.Module):
    def __init__(self,n_embd,n_head,block_size):
        super().__init__()
        self.ln1=nn.LayerNorm(n_embd)
        self.attn=CausalSelfAttention(n_embd,n_head,block_size)
        self.ln2=nn.LayerNorm(n_embd)
        self.mlp=MLP(n_embd)
    def forward(self,x):
        x=x+self.attn(self.ln1(x))
        x=x+self.mlp(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self,vocab_size,n_embd,n_head,n_layer,block_size):
        super().__init__()
        self.token_emb=nn.Embedding(vocab_size,n_embd)
        self.pos_emb=nn.Embedding(block_size,n_embd)
        self.blocks=nn.Sequential(*[Block(n_embd,n_head,block_size) for _ in range(n_layer)])
        self.ln=nn.LayerNorm(n_embd)
        self.lm_head=nn.Linear(n_embd,vocab_size,bias=False)
        self.lm_head.weight=self.token_emb.weight

    def forward(self,idx,targets=None):
        B,T=idx.shape
        pos=torch.arange(T,device=idx.device)
        x=self.token_emb(idx)+self.pos_emb(pos)[None,:,:]
        x=self.blocks(x)
        x=self.ln(x)
        logits=self.lm_head(x)
        loss=None
        if targets is not None:
            loss=F.cross_entropy(logits.view(B*T,-1),targets.view(B*T))
        return logits,loss

    @torch.no_grad()
    def generate(self,idx,max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond=idx[:,-block_size:]
            logits,_=self(idx_cond)
            probs=F.softmax(logits[:,-1,:],dim=-1)
            next_id=torch.multinomial(probs,1)
            idx=torch.cat([idx,next_id],dim=1)
        return idx

model=TinyGPT(vocab_size,n_embd,n_head,n_layer,block_size).to(device)
print("parameters:",sum(p.numel() for p in model.parameters()))


## 5. Sanity check before training

Try to overfit a single batch. If the model cannot drive training loss down on a fixed tiny batch, do not launch a long run.


In [ ]:
fixed_x,fixed_y=get_batch("train")
opt=torch.optim.AdamW(model.parameters(),lr=1e-3)
for step in range(150):
    _,loss=model(fixed_x,fixed_y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    if step%30==0: print(step,float(loss))


## 6. Reset and run the real experiment

The first run should be short enough to finish cheaply. Then extend the token budget only after the curve looks healthy.


In [ ]:
model=TinyGPT(vocab_size,n_embd,n_head,n_layer,block_size).to(device)
optimizer=torch.optim.AdamW(model.parameters(),lr=learning_rate)

@torch.no_grad()
def estimate_loss():
    out={}
    model.eval()
    for split in ["train","val"]:
        vals=[]
        for _ in range(eval_iters):
            x,y=get_batch(split)
            _,loss=model(x,y)
            vals.append(loss.item())
        out[split]=sum(vals)/len(vals)
    model.train()
    return out

t0=time.perf_counter()
for step in range(max_steps):
    if step%eval_interval==0:
        metrics=estimate_loss()
        print(f"step {step:4d} train {metrics['train']:.4f} val {metrics['val']:.4f}")
    x,y=get_batch("train")
    _,loss=model(x,y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
elapsed=time.perf_counter()-t0
if device=="cuda": torch.cuda.synchronize()
print("elapsed seconds:",elapsed)


## 7. Plot learning curves

For a production experiment, log every evaluation point to a structured artifact rather than relying on print output.


In [ ]:
# Minimal extension: rerun with a loss_history list if you want a plot.
# Example plotting pattern:
import matplotlib.pyplot as plt
plt.plot([1,2,3],[3.0,2.0,1.5],marker="o")
plt.xlabel("evaluation step")
plt.ylabel("validation loss")
plt.title("Illustrative validation-loss curve")
plt.show()


## 8. Checkpoint

A real training checkpoint should preserve enough state to resume the experiment, not merely the model weights.


In [ ]:
checkpoint={
    "model":model.state_dict(),
    "optimizer":optimizer.state_dict(),
    "config":{
        "batch_size":batch_size,"block_size":block_size,
        "n_embd":n_embd,"n_head":n_head,"n_layer":n_layer,
        "learning_rate":learning_rate,"max_steps":max_steps,
        "vocab_size":vocab_size
    },
}
torch.save(checkpoint,"tiny_gpt_checkpoint.pt")
print("saved checkpoint bytes:",Path("tiny_gpt_checkpoint.pt").stat().st_size)


## 9. Generate

Generation is inference: the target token is no longer known. The model repeatedly predicts the next token, samples it, appends it, and continues.


In [ ]:
start=torch.tensor([[stoi["
"]]],dtype=torch.long,device=device)
generated=model.generate(start,300)[0].tolist()
print(decode(generated))


## 10. Break the experiment

Change one setting at a time:
- learning rate ×10;
- block size ×2;
- n_embd ×2;
- remove the validation split;
- use a training corpus with heavy duplication.

Before each change, write what you expect.

### Questions

**Why can validation loss increase while training loss keeps decreasing?**

Overfitting, leakage differences, or a mismatch between training and validation distributions can cause it.

**Why does doubling sequence length affect more than memory?**

It changes the amount of context available and increases attention computation; it can also change optimization dynamics and throughput.

**Why is this still not a frontier model?**

Because model/data/compute scale is many orders of magnitude smaller. The point is that the same core training loop scales conceptually into larger systems.

## 11. Research artifact

Fill an experiment card with:
- hypothesis;
- model config;
- token budget;
- hardware;
- throughput;
- losses;
- checkpoint;
- generated examples;
- failure analysis;
- next scaling experiment.

## Next step

The next systems upgrade is efficient attention, mixed precision, gradient accumulation, profiling, and distributed training.

